<a href="https://colab.research.google.com/github/smallrespon/my-resources/blob/main/BERT_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 导入必要的库
# Hugging Face的Transformers库用于BERT模型和分词器，PyTorch用于张量操作，Dataset类用于自定义数据集。
from transformers import BertTokenizer, BertForPreTraining, BertConfig
from transformers import Trainer, TrainingArguments
import torch
import random
from torch.utils.data import Dataset

In [ ]:
# 1. 定义数据集类
# 创建一个自定义数据集类，继承PyTorch的Dataset，用于处理输入文本。
class CustomDataset(Dataset):
    # texts:输入文本列表
    # tokenizer:BERT分词器
    # max_length限制序列长度（BERT-Base推荐128或512）。
    def __init__(self, texts, tokenizer, max_length=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    # 返回数据集的大小，即文本数量。
    def __len__(self):
        return len(self.texts)

    # 准备一个训练样本
    def __getitem__(self, idx):
        # 获取文本对（模拟NSP任务）
        text1 = self.texts[idx]
        # 50% 概率取真实下一句，50% 概率随机采样
        is_next = random.random() < 0.5
        if is_next:
            text2 = self.texts[(idx + 1) % len(self.texts)] #使用模运算，确保索引循环。
            next_sentence_label = torch.tensor([0])  # 正例，为 NSP 任务设置标签为 0
        else:
            # 随机选择一个不是下一句的句子
            random_idx = random.choice([i for i in range(len(self.texts)) if i != idx and i != (idx + 1) % len(self.texts)])
            text2 = self.texts[random_idx]
            next_sentence_label = torch.tensor([1])  # 负例，为 NSP 任务设置标签为 1

        # 对文本进行分词和编码
        encoding = self.tokenizer(
            text1, text2,
            max_length=self.max_length, # 输入序列的最大长度
            padding='max_length', # 补齐到最大长度
            truncation=True,      # 截断到最大长度
            return_tensors='pt'   # 返回PyTorch张量
        )

        # 从分词器返回的 encoding 中提取 input_ids，并移除多余的维度。
        input_ids = encoding['input_ids'].squeeze()
        # 创建 input_ids 的副本，作为 MLM 任务的标签。
        labels = input_ids.clone()

        # 生成一个与 input_ids 形状相同的随机张量，值在 [0, 1) 之间。
        rand = torch.rand(input_ids.shape)
        # 选择 15% 的 token，排除 [CLS] 的token（ID = 101），[SEP] 的token（ID = 102）。[PAD] 的token（ID = 0）。
        mask_arr = (rand < 0.15) * (input_ids != 101) * (input_ids != 102) * (input_ids != 0)

        # 对选中的 15% token，进一步划分 80/10/10
        selected = mask_arr.nonzero(as_tuple=False)  # 获取被选中的 token 位置
        num_selected = selected.size(0) # 计算被选中的 token 数量。
        if num_selected > 0: # 如果有被选中的 token
            # 如果选中多个 token，打乱索引；如果只有一个，保持不变
            if num_selected > 1:
                perm = torch.randperm(num_selected)
                selected = selected[perm]

            # 计算 80%、10%、10% 的数量
            num_mask = int(num_selected * 0.8)  # 80% 掩码
            num_replace = int(num_selected * 0.1)  # 10% 替换
            num_unchanged = num_selected - num_mask - num_replace  # 剩余为不变

            # 从打乱后的 selected 中取出前 num_mask （80%） 个索引，用于掩码。
            mask_indices = selected[:num_mask,0]
            input_ids[mask_indices] = 103  # [MASK]

            # 取出下一段索引，用于随机替换
            replace_indices = selected[num_mask:num_mask + num_replace,0]
            vocab_size = self.tokenizer.vocab_size  # 词汇表大小
            random_tokens = torch.randint(0, vocab_size, replace_indices.shape, dtype=torch.long) # 生成随机 token ID
            input_ids[replace_indices] = random_tokens # 将这些位置的 token 替换为随机值。

            # 10% 保持不变（无需操作，因为 labels 已保留原始值）

        return {
            'input_ids': input_ids, # 修改后的输入序列（包含掩码和随机替换）
            'attention_mask': encoding['attention_mask'].squeeze(), # 注意力掩码，标记有效 token。
            'token_type_ids': encoding['token_type_ids'].squeeze(), # 区分 text1 和 text2 的段落 ID。
            'labels': labels,  # MLM 的原始 token ID。
            'next_sentence_label': next_sentence_label # NSP 标签
        }

In [ ]:
# 2. 准备数据和分词器
texts = [
    "The quick brown fox jumps over the lazy dog.",
    "The dog sleeps peacefully in the sun.",
    "A bright red apple sits on the table."
]
# 从 Hugging Face 的预训练模型库加载 BERT-Base 的分词器
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
# 实例化Dataset，用于生成 BERT 预训练所需的训练样本。
dataset = CustomDataset(texts, tokenizer)

In [ ]:
# 3. 配置BERT-Base参数
config = BertConfig(
    vocab_size=30522, # 词汇表大小。
    hidden_size=768, # 隐藏层维度。
    num_hidden_layers=12, # Transformer层数。
    num_attention_heads=12, # 注意力头数。
    intermediate_size=3072, # 前馈网络维度。
    max_position_embeddings=512 # 最大位置编码。
)

In [ ]:
# 4. 加载预训练模型
# 根据配置初始化BERT预训练模型
model = BertForPreTraining(config=config)

In [ ]:
# 5. 设置训练参数
training_args = TrainingArguments(
    output_dir='./bert_pretraining', # 模型保存路径
    overwrite_output_dir=True, # 覆盖 output_dir 中已有的内容
    num_train_epochs=3, # 训练轮数
    per_device_train_batch_size=2, # 训练时每个设备上的批次大小
    save_steps=10, # 指定每隔多少个训练步骤保存一次模型检查点。过大可能丢失中间结果，过小会增加 IO 开销。
    save_total_limit=2, # 指定最多保留的检查点数量。当保存的检查点数量超过 save_total_limit 时，最旧的检查点会被删除。
    logging_dir='./logs', # 指定训练日志的保存目录。
    logging_steps=5, # 指定每隔多少个训练步骤记录一次日志。
)

In [ ]:
# 6. 初始化Trainer并开始训练
# Trainer是 Hugging Face 提供的一个高级训练工具，用于简化模型训练流程。
# 它封装了训练循环、优化器管理、数据加载和日志记录等功能。
trainer = Trainer(
    model=model,  # 指定要训练的模型
    args=training_args, # 传入训练配置参数
    train_dataset=dataset # 指定训练数据集
)

# 启动模型训练
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: